# [EXP-002] RandomForest + 상황 피처 6개 — 학습

EXP-001 베이스라인을 그대로 유지하면서 현재 투구 전에 알 수 있는 정보로 만든 상황 피처 6개만 추가합니다.

- 비교 기준 Brier Score: 0.248767
- 비교 기준 Validation Score: 416.18
- 학습: 2019~2023년
- 검증: 2024년
- 모델 파라미터: EXP-001과 동일

이 노트북은 베이스라인 파일과 모델을 덮어쓰지 않습니다.

## 1. 라이브러리와 환경 확인

현재 비교 실험은 같은 로컬 환경에서 수행합니다. 최종 제출 모델은 평가 서버와 같은 scikit-learn 1.8.0 환경에서 다시 학습해야 합니다.

In [ ]:
import json
import os
import platform
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__)

## 2. EXP-002 설정

기존 47개 피처에 새로운 피처 6개를 추가합니다.

In [ ]:
DATA_DIR = Path('./data')
ARTIFACT_DIR = Path('./artifacts/EXP-002')
ID = 'row_id'
TARGET = 'control_success'
BASELINE_BRIER = 0.248767
BASELINE_SCORE = 416.18

CAT_COLS = ['top_bottom', 'game_type', 'base_state']
NEW_FEATURES = [
    'count_code',
    'is_full_count',
    'runner_in_scoring_position',
    'same_hand',
    'pitcher_batter_success_gap',
    'pitcher_recent_success_delta',
]

## 3. 피처 생성

이 함수는 학습 코드와 추론 코드에 똑같이 들어가야 합니다. 현재 투구 직전에 제공되는 컬럼만 사용합니다.

In [ ]:
def add_features(df):
    out = df.copy()
    out['count_code'] = out['balls_before'] * 4 + out['strikes_before']
    out['is_full_count'] = (
        (out['balls_before'] == 3) & (out['strikes_before'] == 2)
    ).astype('int8')
    out['runner_in_scoring_position'] = (
        (out['runner_on_2b'] == 1) | (out['runner_on_3b'] == 1)
    ).astype('int8')
    out['same_hand'] = (
        out['pitcher_hand'] == out['batter_hand']
    ).astype('int8')
    out['pitcher_batter_success_gap'] = (
        out['asof_pitcher_success_rate']
        - out['asof_batter_success_rate']
    )
    out['pitcher_recent_success_delta'] = (
        out['asof_pitcher_prev1_game_success_rate']
        - out['asof_pitcher_prev5_game_success_rate']
    )
    return out

## 4. 데이터 로드

기존 베이스라인과 동일하게 test.csv의 컬럼으로 기본 피처 목록을 정한 뒤 피처 6개를 추가합니다.

In [ ]:
test_columns = pd.read_csv(
    DATA_DIR / 'test.csv', encoding='utf-8-sig', nrows=0
).columns
BASE_FEATURES = [column for column in test_columns if column != ID]
FEATURES = BASE_FEATURES + NEW_FEATURES
NUM_COLS = [column for column in FEATURES if column not in CAT_COLS]

train = pd.read_csv(
    DATA_DIR / 'train.csv',
    encoding='utf-8-sig',
    usecols=BASE_FEATURES + [TARGET],
)
train = add_features(train)

print('train:', train.shape)
print('기본 피처:', len(BASE_FEATURES))
print('추가 피처:', len(NEW_FEATURES))
print('전체 피처:', len(FEATURES))

## 5. 전처리와 모델 정의

피처 추가 효과만 비교하기 위해 RandomForest 파라미터는 EXP-001과 동일하게 유지합니다.

In [ ]:
preprocessor = ColumnTransformer([
    (
        'cat',
        OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
        ),
        CAT_COLS,
    ),
    ('num', SimpleImputer(strategy='median'), NUM_COLS),
])

model = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=200,
        n_jobs=-1,
        random_state=42,
    )),
])

## 6. 2024년 검증

Brier Score가 0.248767보다 낮아지고 Validation Score가 416.18보다 높아지는지 확인합니다.

In [ ]:
is_validation = train['season'] == 2024
X_train = train.loc[~is_validation, FEATURES]
y_train = train.loc[~is_validation, TARGET]
X_validation = train.loc[is_validation, FEATURES]
y_validation = train.loc[is_validation, TARGET]

print('train:', len(X_train), '| validation:', len(X_validation))
started_at = time.time()
model.fit(X_train, y_train)
fit_seconds = time.time() - started_at
print(f'학습 완료: {fit_seconds:.1f}초')

started_at = time.time()
validation_predictions = model.predict_proba(X_validation)[:, 1]
inference_seconds = time.time() - started_at

actual_rate = y_validation.mean()
brier = ((validation_predictions - y_validation) ** 2).mean()
baseline_brier = actual_rate * (1 - actual_rate)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f'Brier: {brier:.6f} | 기준선: {baseline_brier:.6f}')
print(f'Validation Score: {score:.2f}')
print(f'EXP-001 대비 Brier: {brier - BASELINE_BRIER:+.6f}')
print(f'EXP-001 대비 Score: {score - BASELINE_SCORE:+.2f}')
print(f'실제 성공률: {actual_rate:.6f}')
print(f'평균 예측 확률: {validation_predictions.mean():.6f}')
print(f'검증 추론 시간: {inference_seconds:.1f}초')

## 7. 검증 결과 저장

실험 결과를 JSON으로 남깁니다.

In [ ]:
metrics = {
    'brier_score': float(brier),
    'baseline_brier': float(baseline_brier),
    'skill_score': float(score),
    'actual_rate': float(actual_rate),
    'prediction_mean': float(validation_predictions.mean()),
    'brier_delta_vs_exp001': float(brier - BASELINE_BRIER),
    'score_delta_vs_exp001': float(score - BASELINE_SCORE),
    'validation_fit_seconds': float(fit_seconds),
    'validation_inference_seconds': float(inference_seconds),
}
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
with (ARTIFACT_DIR / 'validation_metrics.json').open('w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
print('저장 완료:', ARTIFACT_DIR / 'validation_metrics.json')

## 8. 전체 데이터 재학습 및 모델 저장

위 검증 결과가 EXP-001보다 좋아졌을 때만 아래 셀을 실행합니다. 현재 환경 모델은 비교용이며, 실제 제출 모델은 scikit-learn 1.8.0 환경에서 이 셀을 다시 실행해야 합니다.

In [ ]:
SAVE_FINAL_MODEL = False  # 점수가 개선된 것을 확인한 뒤 True로 변경

if SAVE_FINAL_MODEL:
    final_model = Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=200,
            n_jobs=-1,
            random_state=42,
        )),
    ])
    started_at = time.time()
    final_model.fit(train[FEATURES], train[TARGET])
    print(f'전체 재학습 완료: {time.time() - started_at:.1f}초')
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    model_path = ARTIFACT_DIR / 'rf_exp002.pkl'
    joblib.dump(final_model, model_path, compress=3)
    print('모델 저장:', model_path)
else:
    print('최종 모델을 저장하지 않았습니다. 검증 결과를 먼저 비교하세요.')